# 08 — UNIFY (Range-Filtered ANN)

AG News's label is categorical, not a range attribute, so a fair UNIFY-style experiment cannot be run on it directly. Per the paper's own methodology for attribute-less datasets (SIFT1M/GIST1M/Msong/GloVe: a uniform random value in [0, 10000) is attached to each vector), this notebook uses the shared `synthetic_range_attr` column — **not** the AG News label — for all range queries below.

This notebook is self-contained: it installs its own deps, loads data, builds embeddings, and writes its own results CSV. Only the methodology, config constants, seed, and corpus/query construction are shared verbatim across all 9 notebooks in this study.


## 1. Install

In [1]:
# Core install cell. Safe to re-run. Each notebook is independently runnable.
!pip install -q datasets sentence-transformers hnswlib scikit-learn pandas numpy tqdm


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. Imports

In [2]:
import os, time, json, math
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

np.set_printoptions(suppress=True)
pd.set_option("display.max_columns", 50)

import hnswlib


## 3. Shared configuration

In [3]:
# ============================================================
# SHARED CONFIGURATION — identical across all 9 notebooks.
# Only DEBUG_MODE changes corpus/query size; everything else fixed.
# ============================================================
DEBUG_MODE   = True     # small N for correctness checks; set False for full run

N_CORPUS     = 60_000 if not DEBUG_MODE else 3_000
N_QUERIES    = 1_000  if not DEBUG_MODE else 100
K            = 10
SEED         = 42

EMBED_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"   # 384-dim
HNSW_M               = 16
HNSW_EF_CONSTRUCTION = 200
HNSW_EF_SEARCH       = 100
HNSW_SPACE           = "cosine"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"DEBUG_MODE={DEBUG_MODE}  N_CORPUS={N_CORPUS}  N_QUERIES={N_QUERIES}  DEVICE={DEVICE}")


DEBUG_MODE=True  N_CORPUS=3000  N_QUERIES=100  DEVICE=cuda


## 4. Dataset — AG News corpus/query construction (shared, verbatim)

**Synthetic attributes, labeled explicitly:** `synthetic_tag` (uniform(0,1) per corpus doc) is used to carve the 10%/5% sub-selectivity filters out of a single-label 25% filter. `synthetic_range_attr` (uniform int in [0,10000)) has no semantic relationship to the article text and exists only so Notebook 08 can run a range-filtered experiment (AG News's label is categorical, not a range attribute).

In [4]:
# ============================================================
# DATASET — AG News. Hard-coded facts per spec:
#   4 classes, exactly balanced: 0=World, 1=Sports, 2=Business, 3=Sci/Tech
#   train: 120,000 rows (30,000/class). test: 7,600 rows (1,900/class).
# Corpus is drawn from train (subsampled to N_CORPUS), queries from test
# (subsampled to N_QUERIES), using SEED=42 for the subsample. This is why
# natural single-label filters have ~25% selectivity — not an arbitrary
# number, it follows directly from AG News's exact 4-way class balance.
# ============================================================
ds = load_dataset("fancyzhx/ag_news")
assert ds["train"].num_rows == 120_000
assert ds["test"].num_rows == 7_600

rng = np.random.default_rng(SEED)

train_idx_all = np.arange(ds["train"].num_rows)
rng_perm = np.random.default_rng(SEED)
corpus_idx = rng_perm.choice(train_idx_all, size=min(N_CORPUS, len(train_idx_all)), replace=False)
corpus_idx.sort()

test_idx_all = np.arange(ds["test"].num_rows)
rng_perm2 = np.random.default_rng(SEED)
query_idx = rng_perm2.choice(test_idx_all, size=min(N_QUERIES, len(test_idx_all)), replace=False)
query_idx.sort()

corpus_texts  = [ds["train"][int(i)]["text"]  for i in corpus_idx]
corpus_labels = np.array([ds["train"][int(i)]["label"] for i in corpus_idx], dtype=np.int64)

query_texts   = [ds["test"][int(i)]["text"]   for i in query_idx]
query_labels  = np.array([ds["test"][int(i)]["label"] for i in query_idx], dtype=np.int64)

# ------------------------------------------------------------
# SYNTHETIC ATTRIBUTES (explicitly labeled as synthetic; not AG News data).
# synthetic_tag: per-corpus-doc uniform(0,1), used to carve the 10%/5%
#   "synthetic controlled" selectivity filters out of a 25% single-label
#   filter (label==c AND synthetic_tag<0.4 / <0.2).
# synthetic_range_attr: per-corpus-doc uniform integer in [0, 10000), used
#   ONLY by Notebook 08 (UNIFY) to construct a range-filtered ANN
#   experiment, since AG News's label is categorical, not a range
#   attribute. It has NO semantic relationship to the article text —
#   it exists purely to make a range-filtered benchmark possible, mirroring
#   the UNIFY paper's own methodology for attribute-less datasets
#   (SIFT1M/GIST1M: uniform random value in [0, 10000)).
# Both are assigned once here, with a fixed seed, and reused identically
# across every notebook that needs them.
# ------------------------------------------------------------
tag_rng = np.random.default_rng(SEED)
synthetic_tag = tag_rng.uniform(0.0, 1.0, size=len(corpus_idx))

range_rng = np.random.default_rng(SEED)
synthetic_range_attr = range_rng.integers(0, 10_000, size=len(corpus_idx))

corpus_df = pd.DataFrame({
    "corpus_pos": np.arange(len(corpus_idx)),
    "train_idx": corpus_idx,
    "label": corpus_labels,
    "synthetic_tag": synthetic_tag,
    "synthetic_range_attr": synthetic_range_attr,
})
print(corpus_df["label"].value_counts().sort_index())
print(corpus_df.head())


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

label
0    768
1    767
2    704
3    761
Name: count, dtype: int64
   corpus_pos  train_idx  label  synthetic_tag  synthetic_range_attr
0           0         61      2       0.773956                   892
1           1        125      3       0.438878                  7739
2           2        146      3       0.858598                  6545
3           3        182      3       0.697368                  4388
4           4        196      3       0.094177                  4330


## 5. Embedding generation

In [5]:
# ============================================================
# EMBEDDING GENERATION — generated once per notebook, cached in-memory
# (and to /content/*.npy for reuse within the same runtime). Never assume
# a prior notebook already produced these files.
# ============================================================
_model = SentenceTransformer(EMBED_MODEL, device=DEVICE)

def embed(texts, cache_path):
    if os.path.exists(cache_path):
        arr = np.load(cache_path)
        if arr.shape[0] == len(texts):
            return arr
    t0 = time.time()
    arr = _model.encode(
        texts, batch_size=128, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True,
    ).astype(np.float32)
    print(f"Embedded {len(texts)} texts in {time.time()-t0:.2f}s")
    np.save(cache_path, arr)
    return arr

corpus_emb = embed(corpus_texts, "/content/corpus_emb.npy" if os.path.isdir("/content") else "corpus_emb.npy")
query_emb  = embed(query_texts,  "/content/query_emb.npy"  if os.path.isdir("/content") else "query_emb.npy")
print(corpus_emb.shape, query_emb.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Embedded 3000 texts in 4.87s


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedded 100 texts in 0.43s
(3000, 384) (100, 384)


## 6. Filter design (§4, shared)

In [6]:
# ============================================================
# FILTER DESIGN (per spec §4) — the concrete, non-ambiguous scheme every
# notebook uses. "synthetic controlled" filters explicitly use
# synthetic_tag (see markdown above); everything else is a natural label
# filter derived from AG News's exact class balance.
# ============================================================
def build_filters(corpus_df):
    """Return dict: filter_name -> (selectivity_target_pct, boolean mask, type)."""
    labels = corpus_df["label"].values
    tag = corpus_df["synthetic_tag"].values
    filters = {}
    filters["sel100_unfiltered"] = (100, np.ones(len(labels), dtype=bool), "natural")
    filters["sel75_label_ne_3"]  = (75,  labels != 3, "natural")
    filters["sel50_label_in_01"] = (50,  np.isin(labels, [0, 1]), "natural")
    # single-class filters (~25% target) — report all 4 classes' actual results,
    # but use class 0 as the canonical "sel25" filter referenced elsewhere.
    for c in range(4):
        filters[f"sel25_label_eq_{c}"] = (25, labels == c, "natural")
    # synthetic controlled sub-selectivity filters, built ON TOP OF the
    # canonical single-class filter (class 0), per spec.
    filters["sel10_label0_tag_lt_0.4"] = (10, (labels == 0) & (tag < 0.4), "synthetic_controlled")
    filters["sel5_label0_tag_lt_0.2"]  = (5,  (labels == 0) & (tag < 0.2), "synthetic_controlled")
    return filters

FILTERS = build_filters(corpus_df)
filter_summary = []
for name, (target_pct, mask, ftype) in FILTERS.items():
    filter_summary.append({
        "filter": name, "target_selectivity_pct": target_pct,
        "total_corpus_size": len(mask),
        "valid_document_count": int(mask.sum()),
        "actual_selectivity_pct": round(100.0 * mask.sum() / len(mask), 3),
        "type": ftype,
    })
filter_summary_df = pd.DataFrame(filter_summary)
print(filter_summary_df)


                    filter  target_selectivity_pct  total_corpus_size  \
0        sel100_unfiltered                     100               3000   
1         sel75_label_ne_3                      75               3000   
2        sel50_label_in_01                      50               3000   
3         sel25_label_eq_0                      25               3000   
4         sel25_label_eq_1                      25               3000   
5         sel25_label_eq_2                      25               3000   
6         sel25_label_eq_3                      25               3000   
7  sel10_label0_tag_lt_0.4                      10               3000   
8   sel5_label0_tag_lt_0.2                       5               3000   

   valid_document_count  actual_selectivity_pct                  type  
0                  3000                 100.000               natural  
1                  2239                  74.633               natural  
2                  1535                  51.167      

## 7. Ground truth + recall (shared)

In [7]:
# ============================================================
# GROUND TRUTH + RECALL — exact cosine top-K over the filter-valid subset,
# computed identically in every notebook that needs it.
# ============================================================
def exact_topk_filtered(query_vecs, corpus_vecs, mask, k):
    """Exact cosine top-k (corpus_vecs assumed L2-normalized) restricted to
    corpus rows where mask is True. Returns (indices, sims), indices are
    positions into the FULL corpus (not the masked subset)."""
    valid_pos = np.nonzero(mask)[0]
    if len(valid_pos) == 0:
        return np.full((len(query_vecs), k), -1, dtype=np.int64), np.zeros((len(query_vecs), k))
    sub = corpus_vecs[valid_pos]                       # (m, d)
    sims = query_vecs @ sub.T                           # (nq, m) cosine sim (both normalized)
    kk = min(k, sub.shape[0])
    top_local = np.argsort(-sims, axis=1)[:, :kk]
    top_global = valid_pos[top_local]
    if kk < k:
        pad_idx = np.full((len(query_vecs), k - kk), -1, dtype=np.int64)
        top_global = np.concatenate([top_global, pad_idx], axis=1)
    top_sims = np.take_along_axis(sims, top_local, axis=1)
    if kk < k:
        top_sims = np.concatenate([top_sims, np.zeros((len(query_vecs), k - kk))], axis=1)
    return top_global, top_sims

def recall_at_k(retrieved_ids, gt_ids, k=K):
    """retrieved_ids, gt_ids: (n_queries, k) int arrays of corpus positions (-1 = missing)."""
    total = 0.0
    for r, g in zip(retrieved_ids, gt_ids):
        gt_set = set(int(x) for x in g if x >= 0)
        if len(gt_set) == 0:
            total += 1.0  # nothing to find, nothing missed
            continue
        r_set = set(int(x) for x in r if x >= 0)
        total += len(r_set & gt_set) / min(k, len(gt_set)) if len(gt_set) < k else len(r_set & gt_set) / k
    return total / len(retrieved_ids)


## 8. Latency / QPS / results utilities (shared)

In [8]:
# ============================================================
# LATENCY / QPS UTILITIES — query execution time only. Install / embedding /
# preprocessing time is reported separately, never folded into query latency.
# ============================================================
def time_queries(fn, n_repeats=1):
    """fn() runs ALL queries once and returns per-query latencies (list of floats, seconds).
    Caller is responsible for making fn() do only the search, not setup."""
    all_lat = []
    for _ in range(n_repeats):
        lat = fn()
        all_lat.extend(lat)
    arr = np.array(all_lat)
    return {
        "mean_latency_ms": float(arr.mean() * 1000),
        "p50_latency_ms": float(np.percentile(arr, 50) * 1000),
        "p95_latency_ms": float(np.percentile(arr, 95) * 1000),
        "qps": float(1.0 / arr.mean()) if arr.mean() > 0 else float("inf"),
    }

def append_result(rows, method, filter_name, selectivity, recall, lat_stats,
                   index_build_time_s, **extra):
    row = {
        "method": method, "filter": filter_name, "selectivity": selectivity,
        "recall_at_10": recall,
        "mean_latency_ms": lat_stats["mean_latency_ms"],
        "p50_latency_ms": lat_stats["p50_latency_ms"],
        "p95_latency_ms": lat_stats["p95_latency_ms"],
        "qps": lat_stats["qps"],
        "index_build_time_s": index_build_time_s,
    }
    row.update(extra)
    rows.append(row)
    return rows

def save_results(rows, method_slug):
    df = pd.DataFrame(rows)
    path = f"results_{method_slug}.csv"
    df.to_csv(path, index=False)
    print(f"Wrote {path} ({len(df)} rows)")
    return df


## Original UNIFY vs. Our Implementation

**(A)** UNIFY's real contribution: a single graph (Segmented Inclusive Graph / hierarchical HSIG) that provably contains, as a subgraph, the proximity graph from building HNSW over *any* combination of attribute segments — so one index supports pre/post/hybrid filtering with no inconsistency between separately-built indexes, plus range-aware strategy selection (thresholds tau_A/tau_B, empirically calibrated) between pre-filter (skip-list fused into the structure, narrow ranges), post-filter (global HNSW recovered via edge-bitmap, wide ranges), and hybrid/segment-based search (the 'unhappy middle').

**(B)** This notebook reproduces the *three-strategy idea* directly and independently-built (Strategy A: sort+binary-search+brute-force; Strategy B: global HNSW + overfetch + range filter; Strategy C: segment-partitioned HNSW subindexes, S=8 equal-width segments), plus empirically-calibrated range-aware strategy selection between them.

**(C)** What's skipped: SIG/HSIG construction itself, the inclusivity *guarantee* (no proof that segment-combination subgraphs equal the true HNSW-on-that-union), the skip-list fusion for pre-filtering, the bitmap edge-masking for post-filtering, and incremental insertion. Strategy C here is a simplified stand-in for HSIG's segment-and-search idea, not HSIG itself.

**(D vs E):** results below are this notebook's own measurements on this corpus/hardware; no UNIFY paper numbers are copied into `results_unify.csv`.

## 9. Range queries over the synthetic attribute

`synthetic_range_attr` is uniform in [0, 10000) and has **no semantic relationship to the article text** — added only to make a range-filtered experiment possible. Windows `[lo, hi)` are sized to hit target selectivities {90, 75, 50, 25, 10, 5}%.

In [9]:
range_attr = corpus_df["synthetic_range_attr"].values
ATTR_MAX = 10_000
TARGET_SELECTIVITIES = [90, 75, 50, 25, 10, 5]

def window_for_selectivity(pct, attr_max=ATTR_MAX):
    width = attr_max * (pct / 100.0)
    lo = 0  # anchor windows at 0 for reproducibility across notebooks
    hi = min(attr_max, width)
    return lo, hi

range_windows = {}
for pct in TARGET_SELECTIVITIES:
    lo, hi = window_for_selectivity(pct)
    mask = (range_attr >= lo) & (range_attr < hi)
    actual_pct = 100.0 * mask.sum() / len(mask)
    range_windows[f"range_sel{pct}"] = (lo, hi, mask, actual_pct)
    print(f"range_sel{pct:>3d}: window=[{lo},{hi})  actual_selectivity={actual_pct:.2f}%  "
          f"card={int(mask.sum())}")


range_sel 90: window=[0,9000.0)  actual_selectivity=89.40%  card=2682
range_sel 75: window=[0,7500.0)  actual_selectivity=74.53%  card=2236
range_sel 50: window=[0,5000.0)  actual_selectivity=50.20%  card=1506
range_sel 25: window=[0,2500.0)  actual_selectivity=25.17%  card=755
range_sel 10: window=[0,1000.0)  actual_selectivity=9.57%  card=287
range_sel  5: window=[0,500.0)  actual_selectivity=4.63%  card=139


## 10. Strategy A — pre-filter (sort + binary search + brute force)

In [10]:
sort_order = np.argsort(range_attr, kind="stable")
sorted_attr = range_attr[sort_order]

def strategy_a_search(query_vec, lo, hi, k):
    left = np.searchsorted(sorted_attr, lo, side="left")
    right = np.searchsorted(sorted_attr, hi, side="left")
    window_pos = sort_order[left:right]
    if len(window_pos) == 0:
        return []
    sub = corpus_emb[window_pos]
    sims = sub @ query_vec
    top_local = np.argsort(-sims)[:k]
    return list(window_pos[top_local])


## 11. Strategy B — post-filter (global HNSW + overfetch + range filter)

In [11]:
import hnswlib
t0 = time.time()
global_index = hnswlib.Index(space=HNSW_SPACE, dim=corpus_emb.shape[1])
global_index.init_index(max_elements=len(corpus_emb),
                         ef_construction=HNSW_EF_CONSTRUCTION, M=HNSW_M)
global_index.add_items(corpus_emb, np.arange(len(corpus_emb)))
global_build_time_s = time.time() - t0

B_FETCH_K = 250

def strategy_b_search(query_vec, lo, hi, k, fetch_k=B_FETCH_K):
    ef = max(HNSW_EF_SEARCH, fetch_k)
    global_index.set_ef(ef)
    hits, _ = global_index.knn_query(query_vec.reshape(1, -1), k=min(fetch_k, len(corpus_emb)))
    hits = hits[0]
    in_range = [h for h in hits if lo <= range_attr[h] < hi][:k]
    return in_range


## 12. Strategy C — segment-based (simplified stand-in for HSIG), S=8 segments

In [12]:
S_SEGMENTS = 8
seg_width = ATTR_MAX / S_SEGMENTS
seg_bounds = [(int(i * seg_width), int((i + 1) * seg_width)) for i in range(S_SEGMENTS)]

t0 = time.time()
seg_indexes = {}
for si, (slo, shi) in enumerate(seg_bounds):
    seg_mask = (range_attr >= slo) & (range_attr < shi)
    seg_pos = np.nonzero(seg_mask)[0]
    idx = hnswlib.Index(space=HNSW_SPACE, dim=corpus_emb.shape[1])
    eff_M = max(2, min(HNSW_M, len(seg_pos) - 1)) if len(seg_pos) > 1 else 2
    idx.init_index(max_elements=max(1, len(seg_pos)),
                    ef_construction=HNSW_EF_CONSTRUCTION, M=eff_M)
    if len(seg_pos) > 0:
        idx.add_items(corpus_emb[seg_pos], np.arange(len(seg_pos)))
        idx.set_ef(min(HNSW_EF_SEARCH, max(K, len(seg_pos))))
    seg_indexes[si] = (idx, seg_pos)
segment_build_time_s = time.time() - t0
print(f"Built {S_SEGMENTS} segment subindexes in {segment_build_time_s:.2f}s")

def strategy_c_search(query_vec, lo, hi, k):
    touched = [si for si, (slo, shi) in enumerate(seg_bounds) if slo < hi and shi > lo]
    candidates = []
    for si in touched:
        idx, seg_pos = seg_indexes[si]
        if len(seg_pos) == 0:
            continue
        kk = min(max(k, 5 * k), len(seg_pos))
        local_ids, _ = idx.knn_query(query_vec.reshape(1, -1), k=kk)
        global_ids = seg_pos[local_ids[0]]
        candidates.extend(int(g) for g in global_ids if lo <= range_attr[g] < hi)
    if not candidates:
        return []
    cand = np.array(sorted(set(candidates)))
    sims = corpus_emb[cand] @ query_vec
    top_local = np.argsort(-sims)[:k]
    return list(cand[top_local])


Built 8 segment subindexes in 0.51s


## 13. Empirically calibrate tau_A / tau_B (range-aware strategy selection)

Run a handful of sample queries at each selectivity for every strategy, and pick the cardinality crossover points where the cheaper strategy switches — the same way the paper calibrates its thresholds per dataset.

In [13]:
N_CALIB_QUERIES = min(20, len(query_emb))
calib_rows = []
for name, (lo, hi, mask, actual_pct) in range_windows.items():
    card = int(mask.sum())
    for strat_name, strat_fn in [("A", strategy_a_search), ("B", strategy_b_search),
                                  ("C", strategy_c_search)]:
        t0 = time.perf_counter()
        for qi in range(N_CALIB_QUERIES):
            strat_fn(query_emb[qi], lo, hi, K)
        elapsed = time.perf_counter() - t0
        calib_rows.append({"filter": name, "cardinality": card,
                            "strategy": strat_name,
                            "mean_latency_ms": 1000 * elapsed / N_CALIB_QUERIES})
calib_df = pd.DataFrame(calib_rows)
pivot = calib_df.pivot(index="cardinality", columns="strategy", values="mean_latency_ms").sort_index()
print(pivot)

# tau_A: smallest cardinality where A stops being cheapest vs B
# tau_B: smallest cardinality where B becomes cheapest vs C
best_strategy_by_card = pivot.idxmin(axis=1)
print(best_strategy_by_card)

sorted_cards = sorted(pivot.index)
tau_A, tau_B = sorted_cards[0], sorted_cards[-1]
for card in sorted_cards:
    if best_strategy_by_card[card] != "A" and tau_A == sorted_cards[0]:
        tau_A = card
for card in reversed(sorted_cards):
    if best_strategy_by_card[card] == "B" and tau_B == sorted_cards[-1]:
        tau_B = card
print(f"Calibrated thresholds: tau_A~={tau_A} (below: prefer pre-filter), "
      f"tau_B~={tau_B} (above: prefer post-filter)")


strategy            A         B         C
cardinality                              
139          0.073661  1.611772  0.370932
287          0.210339  1.881866  0.411061
755          0.250419  1.556218  1.573456
1506         0.924170  1.293424  1.582984
2236         2.038623  2.981486  5.087502
2682         3.344427  4.083890  6.939680
cardinality
139     A
287     A
755     A
1506    A
2236    A
2682    A
dtype: object
Calibrated thresholds: tau_A~=139 (below: prefer pre-filter), tau_B~=2682 (above: prefer post-filter)


## 14. Full benchmark with range-aware strategy selection

In [14]:
def select_strategy(card, tau_a=tau_A, tau_b=tau_B):
    if card <= tau_a:
        return "A"
    elif card >= tau_b:
        return "B"
    else:
        return "C"

STRAT_FN = {"A": strategy_a_search, "B": strategy_b_search, "C": strategy_c_search}
STRAT_BUILD_TIME = {"A": 0.0, "B": global_build_time_s, "C": segment_build_time_s}

rows = []
for name, (lo, hi, mask, actual_pct) in range_windows.items():
    card = int(mask.sum())
    chosen = select_strategy(card)
    fn = STRAT_FN[chosen]

    gt_ids, _ = exact_topk_filtered(query_emb, corpus_emb, mask, K)

    def run():
        lat = []
        ids_all = np.full((len(query_emb), K), -1, dtype=np.int64)
        for qi in range(len(query_emb)):
            t0 = time.perf_counter()
            found = fn(query_emb[qi], lo, hi, K)
            lat.append(time.perf_counter() - t0)
            ids_all[qi] = (found + [-1] * K)[:K]
        run.last_ids = ids_all
        return lat

    lat_stats = time_queries(run)
    recall = recall_at_k(run.last_ids, gt_ids, K)
    rows.append({
        "method": "unify_range_aware", "filter": name,
        "selectivity": round(actual_pct, 3), "recall_at_10": recall,
        "mean_latency_ms": lat_stats["mean_latency_ms"],
        "p50_latency_ms": lat_stats["p50_latency_ms"],
        "p95_latency_ms": lat_stats["p95_latency_ms"],
        "qps": lat_stats["qps"],
        "index_build_time_s": STRAT_BUILD_TIME[chosen],
        "strategy_chosen": chosen, "cardinality": card,
        "tau_A": tau_A, "tau_B": tau_B,
    })
    print(f"{name:14s} sel={actual_pct:6.2f}%  strategy={chosen}  recall={recall:.3f}")

results_df = pd.DataFrame(rows)
results_df.to_csv("results_unify.csv", index=False)
print(f"Wrote results_unify.csv ({len(results_df)} rows)")
results_df


range_sel90    sel= 89.40%  strategy=B  recall=1.000
range_sel75    sel= 74.53%  strategy=C  recall=1.000
range_sel50    sel= 50.20%  strategy=C  recall=1.000
range_sel25    sel= 25.17%  strategy=C  recall=1.000
range_sel10    sel=  9.57%  strategy=C  recall=1.000
range_sel5     sel=  4.63%  strategy=A  recall=1.000
Wrote results_unify.csv (6 rows)


,method,filter,selectivity,recall_at_10,mean_latency_ms,p50_latency_ms,p95_latency_ms,qps,index_build_time_s,strategy_chosen,cardinality,tau_A,tau_B
0,unify_range_aware,range_sel90,89.400,1.0,1.998913,1.451713,4.275696,500.271938,1.271393,B,2682,139,2682
1,unify_range_aware,range_sel75,74.533,1.0,3.330703,2.734977,6.622443,300.236943,0.506669,C,2236,139,2682
2,unify_range_aware,range_sel50,50.200,1.0,2.340642,1.628069,4.679500,427.233281,0.506669,C,1506,139,2682
3,unify_range_aware,range_sel25,25.167,1.0,0.929377,0.740835,2.714219,1075.989580,0.506669,C,755,139,2682
4,unify_range_aware,range_sel10,9.567,1.0,0.392795,0.390821,0.454240,2545.858550,0.506669,C,287,139,2682
5,unify_range_aware,range_sel5,4.633,1.0,0.061625,0.058317,0.090435,16227.299022,0.000000,A,139,139,2682
